# Лабораторная 2 Численные методы

Вариант 5

Рассмотрим набор различных точек на отрезке $[a,b]:\ x_0 < x_1 < \dots < x_n$, в которых заданы значения функции $f(x)$ так, что $f_i=f(x_i), i=0,1,\dots ,n$. Требуется восстановить значение $f(x)$ и в других точках $x \in [a,b]$.


Для построения таблицы $(x_i, f_i)$ дано:

$[a,b] = [0.35; 1.35]$,  $x_i=0.35 + i \cdot h,\ h=\frac{1}{10},\ i=0,1,\dots,n$, где $n=10$.

$f(x)=0.35 e^x + 0.65 \cos x$


Необходимо:
1. Построить элемент наилучшего приближения методом наименьших квадратов.
2. Построить интерполяционный многочлен Лагранжа.
3. Построить остаток интерполирования в форме Ньютона.
4. Минимизировать остаток интерполирования с помощью многочленов Чебышева.
5. Построить интерполяционный многочлен Ньютона в конце таблицы.

Проверить результат, сравнив истинные значения функции и значения получившегося приближения функции в точках восстановления: $x^*=0.41(6),\ x^{**}=0.9,\ x^{***}=1.31(6)$ и теоретические оценки погрешности.

### Вычислим известные значения функций в узлах $x_i$:

In [18]:
import numpy as np

def foo(x: np.ndarray)->np.ndarray:
    return 0.35 * np.exp(x) + 0.65 * np.cos(x)

a, b, n = 0.35, 1.35, 10

restore_points: np.ndarray = np.array([0.35 + 2/30, 0.9, 1.35 - 1/30])
restore_values = foo(restore_points)

nodes: np.ndarray = np.array([x / 100 for x in range(35, 136, 10)])
values: np.ndarray = foo(nodes)
print(restore_points)
print(values)

[0.41666667 0.9        1.31666667]
[1.10726591 1.13419988 1.1607795  1.18789376 1.21654777 1.24786544
 1.28309239 1.32359907 1.37088435 1.42657957 1.49245328]


|i |0|1|2|
|--|--|--|--|
|x_i |0.35|0.45|0.55|
|f(x_i) |1.1072659053584362| 1.1341998814507992 |  1.160779495592267 |

|i |3|4|5|
|--|--|--|--|
|x_i |0.65|0.75|0.85|
|f(x_i) |1.1878937592117498|1.2165477705824197|1.2478654429993352|

|i |6|7|8|
|--|--|--|-- |
|x_i |0.95|1.05|1.15 |
|f(x_i) |1.2830923889120704| 1.32359907245173 |  1.3708843549661207 |

|i |9|10|
|--|--|--|
|x_i |1.25|1.35|
|f(x_i) | 1.426579570668569| 1.4924532823544179|

## 1. Построение элемента наилучшего приближения методом наименьших квадратов.

Алгоритм:

Строим полином степени $n=5$: $$P(x)= \sum^n_{i=0} c_i x^i$$

Коэффициенты $c_i$ находим из системы уравнений: $$\sum^n_{i=0} c_i (\sum^N_{j=0} p(x_j) x_j^{i+k}) = \sum^N_{j=0} p(x_j) f(x_j) x_j^k,\ k=\overline{0, n},\ p(x) \equiv 1$$
Поскольку $p(x) \equiv 1$, то система является системой линейной алгебраических уравнений. Решим её методом Гаусса.

Найдём погрешность вычислений по формуле $$\Delta f = (\sum_{j=0}^n (f(x_j) - P(x_j))^2)^{1/2}$$

In [19]:
def polynomial(coefficients: np.ndarray, x: np.ndarray)->np.ndarray:
    n = len(x)
    y = np.zeros(n)
    for i in range(len(coefficients)):
        y += coefficients[i] * x ** i
    return y

In [20]:
def gaussian_method(a)->list[float]:
    def change_system_for_select_main_element(system, j):
        max_row = max(system[j:], key=lambda row: row[j])
        max_index = system.index(max_row)
        system[max_index], system[j] = system[j], system[max_index]
        m = 1 if max_index == 0 else -1
        return system, m
    def straight_stroke(a):
        n = len(a)
        for k in range(0,n):
            a, _ = change_system_for_select_main_element(a,k)
            for j in range(n,k-1,-1):
                a[k][j] /= a[k][k]
            for i in range(k+1,n):
                for j in range(n,k-1,-1):
                    a[i][j] = a[i][j] - a[i][k]*a[k][j]
        return a
    def reverse_stroke(a: list[list[float]])->list[float]:
        n = len(a)
        x = [0.] * n
        x[n-1] = a[n-1][n]
        for i in range(n-2,-1,-1):
            s = 0
            for j in range(i+1,n):
                s+=a[i][j]*x[j]
            x[i] = a[i][n] - s
        return x
    
    return reverse_stroke(straight_stroke(a))

In [21]:
import numpy as np

def least_squared_method(nodes: np.ndarray,
                          function_nodes: np.ndarray, degree: int)->np.ndarray:
    m = degree + 1
    matrix = np.zeros((m, m+1))
    for i in range(m):
        for j in range(m):
            matrix[i][j] += (nodes ** (i+j)).sum()
        matrix[i][m] = (function_nodes * nodes ** i).sum()
    
    coefficients = np.array(gaussian_method(matrix.tolist()))
    return coefficients

least_squared_coeffs = least_squared_method(nodes, values, 5)
print("Coefficients of least squared polynomial:")
print(least_squared_coeffs)

print("Calculation error:")
least_squared_calc_error = ((values - polynomial(
    least_squared_coeffs, nodes)) ** 2).sum() ** (1/2)
print(least_squared_calc_error)

print("Error in recovery points:")
practical_squared_theoretical_error = polynomial(
    least_squared_coeffs, restore_points) - restore_values
for i in range(len(practical_squared_theoretical_error)):
    print(practical_squared_theoretical_error[i], end=' ')
print()
print(np.sum(practical_squared_theoretical_error ** 2) ** (1/2))

Coefficients of least squared polynomial:
[ 0.99992389  0.35061959 -0.15191199  0.0610772   0.04001999  0.00286639]
Calculation error:
6.45945564733815e-07
Error in recovery points:
3.3968199075751215e-07 2.133482932542563e-07 3.578693053007953e-07 
5.375609628274402e-07


Таким образом, многочлен, полученный методом наименьших квадратов, выглядит следующим образом:
 $$\begin{aligned}P(x) = 0.99992389 + 0.35061959 x -0.15191199  x^2+ \\ + 0.0610772 x^3 + 0.04001999 x^4 + 0.0.00286639 x^5 \end{aligned}$$

Погрешность вычислений: $\Delta f = 6.45945564733815e-07$. Невязка многочлена в заданных точках $x^*, x^{**}, x^{***}:$

|Формула|Значение|
|------|----|
|$P(x^*)-f(x^*)$| $3.3968199075751215e-07$|
|$P(x^{**})-f(x^{**})$ | $2.133482932542563e-07$ |
|$P(x^{***})-f(x^{***})$ | $3.578693053007953e-07 $ |

Норма невязки равна $5.375609628274402e-07$.

С помощью метода наименьших квадратов найдено приближённое значение функции в заданных точках.

## 2. Построение интерполяционного многочлена Лагранжа

Интерполяционный многочлен Лагранжа имеет вид:$$P_n(x) = \sum_{i=0}^n l_i(x) f(x_i)$$ $$l_i=\frac{(x-x_0) \dots (x-x_{i-1})(x-x_{i+1}) \dots (x-x_n))}{(x_i-x_0) \dots (x_i-x_{i-1})(x_i-x_{i+1}) \dots (x_i-x_n)}$$ Если определить $\omega_{n+1}(x) = (x-x_0) \dots (x-x_n)$ - многочлен степени $(n+1)$, тогда можно записать многочлен Лагранжа как: $$P_n(x) = \sum_{i=0}^0 \frac{\omega_{n+1}(x)}{(x-x_i)\omega_{n+1}'(x_i)}f(x_i)$$

In [22]:
def lagrange_polynomial(nodes: np.ndarray, function_nodes: np.ndarray, 
                        x: np.ndarray)->np.ndarray:
    P_arr = np.zeros(len(x))
    for idx, x_val in enumerate(x):
        total = 0.0
        for i, xi in enumerate(nodes):
            term = function_nodes[i]
            for j, xj in enumerate(nodes):
                if i == j:
                    continue
                term *= (x_val - xj) / (xi - xj)
            total += term
        P_arr[idx] = total
        
    return P_arr

print("Error in recovery points:")
lagrange_error = np.abs(lagrange_polynomial(nodes, values,
                     restore_points) - restore_values)
for i in lagrange_error:
    print(i)

Error in recovery points:
6.150635556423367e-14
2.220446049250313e-15
1.3655743202889425e-13


Невязка многочлена в заданных точках $x^*, x^{**}, x^{***}:$

|Формула|Значение|
|------|----|
|$P(x^*)-f(x^*)$| $6.150635556423367e-14$|
|$P(x^{**})-f(x^{**})$ | $2.220446049250313e-15$ |
|$P(x^{***})-f(x^{***})$ | $1.3655743202889425e-13$ |

С помощью интерполяционного многочлена Лагранжа найдено приближённое значение функции в заданных точках и погрешность интерполирования.

## 3. Построение остатка интерполирования в форме Ньютона

Для представления остатка интерполирования в форме Ньютона необходимо ввести понятие разделённых разностей. Разделённые разности - дискретный аналог производный. Рассмотрим узлы $x_i,\ i=\overline{1,n}$. Разделённая разность нулевого порядка совпадает со значением $f(x_i)$. Разделённая разность $(k+1)$-го порядка определяется формулой: $$f(x_0, \dots, x_{k+1}) = \frac{f(x_1,\dots, x_{k+1}) - f(x_0, \dots, x_k)}{x_{k+1}- x_0}$$ Остаток интерполирования в форме Ньютона определяется формулой: $$r_n(x) = \omega_{n+1} (x) f(x, x_0, \dots, x_n)$$ где $\omega_{n+1} (x)=(x-x_0) \dots (x-x_n),\ f(x, x_0, \dots, x_n)$ - разделённая разность $(n+1)$-го порядка.

Для вычисления разделённых разностей нужны значения функции в точках $x^*, x^{**}, x^{***}$. Для их вычисления в рамках лабораторной работы воспользуемся истинным видом функции.

Для реализации подсчёта разделённых разностей воспользуемся верхнетреугольной матрицей.

In [23]:
def calculate_divided_difference(nodes: np.ndarray, 
                                 function_nodes: np.ndarray)->np.ndarray:
    n = len(nodes)
    table = np.zeros((n, n))
    table[:, 0] = function_nodes
    for i in range(1, n):
        for j in range(n-i):
            table[j][i] = (table[j+1][i-1] - table[j][i-1]) / (
                                        nodes[j+i] - nodes[j])
    return table

def interpolation_remainder_newton_form(nodes: np.ndarray, values: np.ndarray,
                                         x: float, foo_x: float)->float:
    n = len(nodes)
    new_nodes = np.append([x], nodes)
    new_values = np.append([foo_x], values)
    remainder = calculate_divided_difference(new_nodes, new_values)[0][n]
    for i in range(len(nodes)):
        remainder = remainder * (x - nodes[i])

    return remainder

for x, f in zip(restore_points, restore_values):
    print(np.abs(interpolation_remainder_newton_form(nodes, values, x, f)))


6.097693331869179e-14
1.71312870710646e-15
1.3965520118204129e-13


|Формула|Значение|
|------|----|
|$r(x^*)$| $6.097693331869179e-14$|
|$r(x^{**})$ | $1.71312870710646e-15$ |
|$r(x^{***})$ | $1.3965520118204129e-13$ |

Сравним теоретические остатки в форме Ньютона с фактическими (реальными) остатками построенного методом Лагранжа интерполяционного многочлена. Степени среднеквадратических ошибок совпадают, отличается третья цифра. Как видно, теоретическая погрешность в первой и второй точках восстановления оказалась больше, чем реальная погрешность в этих же точках. Связано это с умножением малых величин и делением на малые величины (при подсчёте разделённых разностей). Из-за этого накопилась вычислительная погрешность и истинные погрешности отличаются от теоретических.

## 4. Минимизация остатка интерполирования при помощи многочленов Чебышева.

Многочлены Чебышева: $$T_0(x)=1,\ T_1(x)=x,\ T_{n+1}(x)=2xT_n(x)-T_{n-1}(x),\ n=1,2,\dots$$

Тригонометрическая форма представления многочленов Чебышева: $T_n(x)=\cos (n \arccos x),\ n \ge 0,\ x \in [-1,1]$

Приведённые многочлены Чебышева: $\overline{T}_n(x) = 2^{1-n} T_n(x)$

> ##### Теорема
> Если $P_n(x)$ - это многочлен степени $n$ со старшим коэффициентом равным единице, то $$\max_{x \in [-1, 1]} |P_n(x)| \ge \max_{x \in [-1, 1]} |\overline{T}_n(x)| \ge 2^{1-n}$$

Таким образом, если выбрать в качестве узлов интерполирования корни многочлена Чебышева, то остаток интерполирования будет минимальным. Корни этого многочлена будут равны: $$x_k = \frac{a+b}{2} + \frac{b-a}{2} \cos \frac{(2k+1) \pi}{2(n+1)},\ k=\overline{0,n}$$
Максимальное значение этого отклонения равно: $$\max_{x \in [a,b]} |\omega_{n+1}(x)| = \frac{(b-a)^{n+1}}{2^{2n+1}}$$ Значит: $$|r_n(x)| \le \frac{M}{(n+1)!} \cdot \frac{(b-a)^{n+1}}{2^{2n+1}}$$ где $M \ge |f^{(n+1)}(x)|,\ x \in [a,b]$

Также нужно перенумеровать узлы: $\widetilde{x}_k = x_{n-k}$.

Вычислим новые узлы интерполирования как корни многочлена Чебышева $(n+1)$-ой степени и построим интерполяционный многочлен Лагранжа на полученные узлах.

Вычислим $M$ для исходной функции: $f^{(n+1)}(x) = f^{(11)}(x) = f'''(x) = 0.35e^x + 0.65 \sin x$. Найдём максимальное значение на отрезке $[0.35; 1.35]$. Функция - возрастающая на промежутка, значит, нужно найти значение в точке $1.35$: $f'''(1.35) = 1.98433$

In [24]:
import math
M = 1.98433

def chebyshev_roots(a: float, b: float, n: int)->np.ndarray:
    k = np.flip(np.arange(n + 1))
    cos_arg = (2*k + 1) * np.pi / (2*(n + 1))
    roots = (a + b)/2 + (b - a)/2 * np.cos(cos_arg)
    return roots

def chebyshev_remainder(a: float, b: float, n :int, M: float):
    return M / math.factorial(n+1) * (b-a)**(n+1) / 2**(2*n+1)

chebyshev_nodes = chebyshev_roots(a, b, n)
chebyshev_values = foo(chebyshev_nodes)

print(chebyshev_nodes)

print("Error in recovery points:")
lagrange_error = lagrange_polynomial(chebyshev_nodes, chebyshev_values,
                                                     restore_points) - restore_values
for i in lagrange_error:
    print(i)

print("Theoretical Chebyshev error:")
print(chebyshev_remainder(a,b,n,M))

[0.35508928 0.395184   0.47212521 0.57967959 0.70913372 0.85
 0.99086628 1.12032041 1.22787479 1.304816   1.34491072]
Error in recovery points:
1.3100631690576847e-14
1.4210854715202004e-14
1.0436096431476471e-14
Theoretical Chebyshev error:
2.370436202644518e-14


|Формула|Значение|
|------|----|
|$P(x^*)-f(x^*)$| $1.3100631690576847e-14$|
|$P(x^{**})-f(x^{**})$ | $1.4210854715202004e-14$ |
|$P(x^{***})-f(x^{***})$ | $1.0436096431476471e-14$ |

Оценка погрешности Чебышева: $2.370436202644518e-14$

В первой и третьей точках восстановления погрешность уменьшилась по сравнению с методом Лагранжа на равностоящих узлах. Но второй точке погрешность увеличилась. Это кажется противоречием, но минимизация многочлена Чебышева гарантирует минимизацию максимальной погрешности на всём отрезке, а не в каждой точке. 

Почему так произошло? Связано это с тем, что узлы Чебышева сгущаются к концам отрезка, в середине их меньше. Поэтому в центральной области интерполяция может быть менее точной, чем на равномерной сетке.

С помощью многочленов Чебышева были найдены узлы, при помощи которых была минимизирована максимальная погрешность интерполяционного многочлена Лагранжа.

## 5. Построение интерполяционного многочлена Ньютона в конце таблицы.

Построим многочлен третьей степени. Значит, для интерполирования нужно четыре точки. Для этого возьмём четыре последние точки: $x_0 = 1.05,\ x_i=x_0 + ih,\ i=\overline{0,3}, h=\frac{1}{10}$

Функция $f(x)$ задана таблично в равностоящих точках $x_i = x_0 + ih$.

Определим конечные разности: Конечная разность нулевого порядка совпадает со значением функции $f(x_i)=f_i$, конечная разность первого порядкам определяется $\Delta f_i = f_{i+1} - f_i$. Конечная разность $k$-го порядка определяется $\Delta^k f_i = \Delta^{k-1} f_{i+1} - \Delta^{k-1} f_i$. Аппарат конечных разностей является частным случаем аппарата разделённых разностей.

Зададим интерполяционный многочлен Ньютона для конца таблицы: $$ \begin{gathered} P_k(x) = P_k(x_n + th) = f_n + \frac{t}{1!} \Delta f_{n-1} + \frac{t(t+1)}{2!} \Delta^2 f_{n-2} + \\ + \dots + \frac{t(t+1)\dots(t+k-1)}{k!} \Delta^k f_{n-k} \end{gathered}$$

Интерполировать будем в точке $x^{***} = 1.31(6)$. $t = \frac{x-x_n}{h} = -0.(3)$

Оценка погрешности: $$r_k(x)=r_k(x_n+th)=h^{k+1} \frac{t(t+1)\dots(t+k)}{(k+1)!}f^{(k+1)}(\xi),\ \xi \in [x_n,x_n-kh]$$

$$f^{(k+1)}(\xi) = f^{(4)}(\xi) = 0.35 e^x + 0.65 \cos x$$

Производная - возрастающая на всём промежутке, значит, максимально её значение достигается в точке $x = 1.35$: $\max f^{(4)}(\xi) = 1.49245$

In [29]:
def finite_differences(nodes: np.ndarray, values: np.ndarray)->np.ndarray:
    n = len(nodes)
    matrix = np.zeros((n, n))
    matrix[:, 0] = values
    
    for i in range(1, n):
        for j in range(n-1, i-1, -1):
            matrix[j][i] = matrix[j][i-1] - matrix[j-1][i-1]
    
    return matrix[-1, :]

def interpolation_table_end(nodes: np.ndarray, values: np.ndarray,
                                                     x: float)->float:
    fin_diff = finite_differences(nodes, values)
    h = nodes[1] - nodes[0]
    t = (x - nodes[-1]) / h
    print("t:", t)
    value = 0.0
    k = len(fin_diff)
    for i in range(k):
        temp = 1
        for j in range(i):
            temp *= (t + j)
        value += fin_diff[i] * temp / math.factorial(i)

    return value

def theoretical_reminder_table_end(x: float, last_node: float, step: float, k : int, M: float):
    ans = M * (step ** (k+1))
    t = (x - last_node) / step
    for i in range(k+1):
        ans *= (t + i)
    ans /= math.factorial(k + 1)
    return ans

table_nodes = nodes[-4:]
table_values = values[-4:]

print("Error in recovery point:")
r = np.abs(restore_values[2] - interpolation_table_end(table_nodes, 
                                table_values, restore_points[2]))
print(r)

print("Theoretical error:")
print(theoretical_reminder_table_end(restore_values[2], nodes[-1], 1/10, 4, 1.49245))

Error in recovery point:
t: -0.3333333333333348
5.8098070496725995e-06
Theoretical error:
2.259915404120564e-05


Погрешность: $5.8098070496725995e-06$

Теоретическая ошибка в точке $x^{**}$: $2.259915404120564e-05$

С помощью интерполирования в конце таблицы было получено значение в точке $x^{***}$ со степенью погрешности $-6$, поскольку использовалось всего 4 узла. Если использовать все 11, то получится такая же погрешность, как у метода Ньютона. Это напрямую связано с выводом формулы для интерполирования в конце таблицы.

## Выводы

|Метод|Норма невязки|
|------|----|
|Метод наименьших квадратов| $5.375609628274402e-07$|
|Многочлен Лагранжа | $1.4978616223729628e-13$ |
|Минимизация остатка | $2.196558788550403e-14$ |
|В конце таблицы | $-5.8098070496725995e-06$ |

Наилучшая погрешность получилась при минимизации остатка интерполяционного многочлена Лагранжа. Это связано с тем, что при реализации этого метода строился многочлен 10-й степени. Многочлен Лагранжа без минимизации остатка оказался на порядок худше, чем с минимизацией. Метод наименьших квадратов показал результат хуже, поскольку строился многочлен пятой степени. Самым худшим оказался многочлен Ньютона в конце таблицы, так как для него использовалось всего четыре узла и получился многочлен третьей степени.

Таким образом, исследованы некоторые методы для решения проблемы построения приближения функций.